# HEO Na 층상 양극재 MLIP 스크리닝 v3 — RunPod 멀티 GPU

**엔진은 v1**(7,279조성 완주 검증: 멀티프로세싱 SQS, torch-sim 배치 + 구조별 폴백, CSV 체크포인트, GPU당 샤드),
**물리·장부는 v2 스펙**(CLAUDE.md: 정확 전하중성, 규칙 기반 O3→P3 glide, 짝지은 빈자리, meV/f.u.).

실행 순서 — 위에서 아래로. 단 **셀 6·9 다음에는 셀 7·10이 "PASS n COMPLETE"를 찍을 때까지 반복 실행**:

```
1  CONFIG        1b 워치독        2  INSTALL       3  워커 확인
4  조성 열거      5  ★ 앵커 게이트 (실패하면 여기서 멈춤)
6  PASS 1 런처  →  7  모니터
8  Na 참조 + hull + Ehull_eff / T*
9  PASS 2 런처  →  10 모니터
11 집계 · δ · 상 판정 · 채점 · 불변량 · audit      12 top-N 내보내기      13 아카이브 · 종료
```

## 셀 1 — 설정

In [1]:
import os
os.environ["HEO_TERMINATE_MODE"] = "none"
os.environ["MP_API_KEY"] = "0md9oV9ieH21W4cmejXD6YQZHz6rRv7F"
MP_API_KEY = os.environ["MP_API_KEY"]

In [2]:
# ============ CELL 1: CONFIG ============
# Single source of truth is heo_worker.py; this cell only sets the environment it reads.
import os, json, subprocess, sys, time
from pathlib import Path

os.environ.setdefault("HEO_WORKDIR",     "/workspace/heo_v2")   # Network Volume: survives a pod delete
os.environ.setdefault("HEO_MODE",        "orb")                 # orb | mace | mock
os.environ.setdefault("HEO_SQSCUTOFFS",  "6.0,4.0")             # v1-verified cluster cutoffs
os.environ.setdefault("HEO_SQSSTEPS",    "3000")
os.environ.setdefault("HEO_EHULLCUT",    "0.10")                # on Ehull_eff (entropy-corrected)
os.environ.setdefault("HEO_SAVE_RELAXED","1")                   # every relaxed cell -> CIF, so no re-relax later
os.environ.setdefault("HEO_CHUNK",       "100")                 # compositions per SQS+relax cycle
os.environ.setdefault("HEO_TERMINATE_MODE", "none")             # stop | delete | none  (user 2026-09-03: never auto-terminate)

WORKDIR   = os.environ["HEO_WORKDIR"]
MODE      = os.environ["HEO_MODE"]
POD_ID, N_PODS = 0, 1                                  # only for splitting one run across several pods
LIMIT     = int(os.environ.get("HEO_LIMIT", 0))        # smoke-test cap on #compositions; 0 = all
MAX_HOURS = float(os.environ.get("HEO_MAX_HOURS", 36))
IDLE_MIN  = float(os.environ.get("HEO_IDLE_MIN", 60))

# MP_API_KEY comes from the shell (export MP_API_KEY=...). Never write the key into this notebook:
# a notebook is shared and committed far more often than a shell session.
MP_API_KEY = os.environ.get("MP_API_KEY", "")

N_GPUS = 0
try:
    import torch; N_GPUS = torch.cuda.device_count()
except Exception:
    try:      # no torch yet (first run, before cell 2) -> ask the driver instead
        _o = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
        N_GPUS = sum(1 for l in _o.splitlines() if l.startswith("GPU "))
    except (FileNotFoundError, OSError):
        N_GPUS = 0                       # no driver either (local / mock box)
N_GPUS = int(os.environ.get("HEO_NGPU", 0)) or N_GPUS or 1

for sub in ("", "/structures/sqs", "/structures/relaxed", "/checkpoints", "/results", "/hull_cache"):
    os.makedirs(WORKDIR + sub, exist_ok=True)
Path(WORKDIR, "results", "DONE").unlink(missing_ok=True)   # a stale sentinel must not fire the watchdog

print(f"MODE={MODE} | GPUs={N_GPUS} | workdir={WORKDIR} | LIMIT={LIMIT or 'all'}")
print(f"MP_API_KEY: {'set' if MP_API_KEY else 'NOT SET -> cell 8 will fail'}")
print(f"terminate={os.environ['HEO_TERMINATE_MODE']} | max {MAX_HOURS} h | idle {IDLE_MIN} min")
if MODE != "mock":
    for i in range(N_GPUS):
        try: print(" ", torch.cuda.get_device_name(i))
        except Exception: pass

MODE=orb | GPUs=1 | workdir=/workspace/heo_v2 | LIMIT=all
MP_API_KEY: set
terminate=none | max 36.0 h | idle 60.0 min
  NVIDIA RTX PRO 4500 Blackwell


## 셀 1b — 팟 과금 방지 워치독

노트북 셀은 **첫 예외에서 멈춥니다.** 앵커 게이트나 어떤 assert가 걸리면 셀 13에 도달하지 못하고, 커널은 살아 있으니 팟이 계속 과금됩니다.
이 셀은 커널과 무관하게 사는 프로세스를 띄워 ① 하드 데드라인 ② 파일 무활동 + GPU 유휴 ③ `DONE` 이후에도 팟이 살아 있음, 세 경우에 팟을 내립니다.

v1의 "워커 PID 전원 종료 → 삭제"는 **워커가 행(hang)하면 영원히 안 걸리고**, 반대로 PASS 1이 정상 종료한 직후에도 걸려서 hull 셀 전에 팟을 죽입니다.

In [3]:
# ============ CELL 1b: pod cost guard — detached watchdog ============
# A notebook cell stops at the FIRST exception, so cell 13 is NOT reached when any assert fires
# (anchor gate, hull, invariants). The kernel then sits idle and the pod bills forever. This starts
# an independent process that outlives the kernel and terminates the pod on any of:
#   (1) hard deadline, (2) no file activity for IDLE_MIN while the GPUs are idle,
#   (3) results/DONE written by cell 13 but the pod still alive 5 min later.
# v1 terminated on "all worker PIDs exited", which never fires if a worker hangs and fires too
# early if it exits between passes (killing the pod before the hull cell). Set HEO_WATCHDOG=0 to disable.
import textwrap
WATCHDOG_SRC = textwrap.dedent(r"""
    import os, sys, time, subprocess, urllib.request
    from pathlib import Path
    WORKDIR = Path(sys.argv[1]); MAX_H = float(sys.argv[2]); IDLE_MIN = float(sys.argv[3]); MODE = sys.argv[4]
    LOG = open(WORKDIR / "watchdog.log", "a", buffering=1)
    def say(m): LOG.write(f"[{time.strftime('%H:%M:%S')}] {m}\n")

    # runpodctl first, REST second. Never raises: a missing binary must not skip the fallback.
    def pod_terminate(mode):
        pod = os.environ.get("RUNPOD_POD_ID")
        if not pod: return False, "RUNPOD_POD_ID not set"
        cmds = ([["runpodctl","pod","delete",pod],["runpodctl","remove","pod",pod]] if mode == "delete"
                else [["runpodctl","pod","stop",pod],["runpodctl","stop","pod",pod]])
        for c in cmds:
            try:
                if subprocess.run(c, capture_output=True).returncode == 0: return True, " ".join(c)
            except Exception as e: say(f"  {c[0]} unusable: {e!r}")
        key = os.environ.get("RUNPOD_API_KEY")
        if not key: return False, "runpodctl failed and RUNPOD_API_KEY not set"
        url = f"https://rest.runpod.io/v1/pods/{pod}" + ("" if mode == "delete" else "/stop")
        req = urllib.request.Request(url, method="DELETE" if mode == "delete" else "POST",
                                     headers={"Authorization": f"Bearer {key}"})
        try:
            urllib.request.urlopen(req, timeout=20); return True, "REST " + url
        except Exception as e: return False, f"REST failed: {e!r}"

    def gpus_busy():
        try:
            out = subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu","--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=30).stdout
            return any(int(x) > 5 for x in out.split() if x.strip().isdigit())
        except Exception:
            return False          # no nvidia-smi -> fall back to file activity alone

    def snapshot():
        s = []
        for p in WORKDIR.rglob("*"):
            if p.is_file() and p.name != "watchdog.log":
                try: s.append((str(p), p.stat().st_size, int(p.stat().st_mtime)))
                except OSError: pass
        return hash(tuple(sorted(s)))

    def fire(mode, why):
        if mode == "none":                       # terminate disabled by the user: log what WOULD happen
            say(f"LOG-ONLY - would terminate now: {why}")
            return
        say(f"TERMINATING ({mode}) - {why}")
        ok, how = pod_terminate(mode)
        say(f"  -> {'OK ' + how if ok else 'FAILED: ' + how}")
        sys.exit(0 if ok else 1)

    say(f"watchdog up | max {MAX_H} h | idle {IDLE_MIN} min | mode {MODE} | pod {os.environ.get('RUNPOD_POD_ID')}")
    t0 = time.time(); last_sig = snapshot(); last_change = time.time(); done_at = None
    while True:
        time.sleep(60)
        if (WORKDIR / "results" / "DONE").exists():
            done_at = done_at or time.time()
            if time.time() - done_at > 300:
                fire(MODE, "DONE written, pod still up after 5 min"); done_at += 21600   # re-log every 6 h in log-only
            continue
        if time.time() - t0 > MAX_H * 3600:
            fire(MODE, f"hard deadline {MAX_H} h"); t0 += 21600
        sig = snapshot()
        if sig != last_sig: last_sig, last_change = sig, time.time(); continue
        idle_s = time.time() - last_change
        if idle_s > IDLE_MIN * 60:
            if gpus_busy():
                say(f"  files idle {idle_s/60:.0f} min but GPUs busy - holding"); last_change = time.time()
            else:
                fire(MODE, f"no file activity for {idle_s/60:.0f} min and GPUs idle")
                last_change = time.time()        # log-only: re-arm instead of spamming every minute
""")

_wd_mode = os.environ["HEO_TERMINATE_MODE"]
# "none" now passes through unchanged: the watchdog still runs but only LOGS what it would have
# done (user decision 2026-09-03: this run must never be terminated automatically).
if os.environ.get("HEO_WATCHDOG", "1") == "1" and os.environ.get("RUNPOD_POD_ID"):
    _wd = Path(WORKDIR, "pod_watchdog.py"); _wd.write_text(WATCHDOG_SRC)
    _p = subprocess.Popen([sys.executable, str(_wd), WORKDIR, str(MAX_HOURS), str(IDLE_MIN), _wd_mode],
                          env=dict(os.environ), start_new_session=True,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"pod watchdog started (pid {_p.pid}) -> max {MAX_HOURS} h, idle {IDLE_MIN} min, "
          f"mode {_wd_mode}, log {WORKDIR}/watchdog.log")
    if _wd_mode == "none":
        print("  terminate DISABLED -> log-only watchdog; the pod BILLS until you stop it yourself")
elif not os.environ.get("RUNPOD_POD_ID"):
    print("pod watchdog: RUNPOD_POD_ID not set -> not on a RunPod pod, nothing to guard")
else:
    print("pod watchdog DISABLED (HEO_WATCHDOG=0)")

pod watchdog started (pid 505) -> max 36.0 h, idle 60.0 min, mode none, log /workspace/heo_v2/watchdog.log
  terminate DISABLED -> log-only watchdog; the pod BILLS until you stop it yourself


## 셀 2 — 패키지 설치

In [4]:
# ============ CELL 2: INSTALL ============
# mace-torch is only needed for the contrast run (HEO_MODE=mace, separate HEO_WORKDIR).
!pip install -q pymatgen mp-api icet ase torch-sim-atomistic orb-models
import torch_sim, orb_models, icet
print("torch-sim OK | orb-models OK | icet OK")

torch-sim OK | orb-models OK | icet OK


## 셀 3 — 워커 모듈 확인

`heo_worker.py`를 이 노트북과 **같은 디렉터리에 업로드**해 두세요. 임포트되는지, 템플릿 검증(적층 글자 + 프리즘 질서변수)이 통과하는지, 어떤 파일이 로드됐는지 md5로 확인합니다.

In [7]:
# ============ CELL 3: worker module check ============
import hashlib, importlib, sys
sys.path.insert(0, os.getcwd())
import heo_worker as W; importlib.reload(W)
print("heo_worker.py:", W.__file__)
print("  md5:", hashlib.md5(open(W.__file__, "rb").read()).hexdigest())
print("  model tag:", W.MODEL_TAG, "| SQS cutoffs:", W.SQS_CUTOFFS, "| MC steps:", W.SQS_MC_STEPS)
print("  lattice a,c,z_O =", W.A_HEX, W.C_HEX, W.Z_O,
      "|", W.N_ATOMS, "atoms | remove", W.N_NA_REMOVE, "Na |", W.N_VAC_SAMPLES, "vacancy samples")
W._validate_templates(verbose=True)     # O3 CABCAB / P3 CAABBC, prismatic order -1 / +1, min Na-O > 2 A

heo_worker.py: /workspace/heo_worker.py
  md5: 8a8fe00946ac664484b58a860a132fbe
  model tag: orb_v3_conservative_inf_mpa | SQS cutoffs: [6.0, 4.0] | MC steps: 3000
  lattice a,c,z_O = 2.96 15.95 0.235 | 108 atoms | remove 13 Na | 5 vacancy samples
O3 O-layer letters: CABCAB | Na gaps: ['O', 'O', 'O']
P3 O-layer letters: CAABBC | Na gaps: ['P', 'P', 'P'] | glide: (0.2222222222222222, 0.1111111111111111)
O3: min Na-O = 2.320 A | prismatic order P = -1.0000 over 27 Na (expect -1)
P3: min Na-O = 2.320 A | prismatic order P = +1.0000 over 27 Na (expect +1)


Structure Summary
Lattice
    abc : 8.879999999999999 8.88 15.95
 angles : 90.0 90.0 119.99999999999999
 volume : 1089.2241219228654
      A : 8.879999999999999 0.0 5.437431788214248e-16
      B : -4.4399999999999995 7.690305585605817 5.437431788214248e-16
      C : 0.0 0.0 15.95
    pbc : True True True
PeriodicSite: Na (1.48, 2.563, 10.63) [0.3333, 0.3333, 0.6667]
PeriodicSite: Na (-1.48e-10, 5.127, 10.63) [0.3333, 0.6667, 0.6667]
PeriodicSite: Na (2.96, 8.545e-11, 10.63) [0.3333, 1.111e-11, 0.6667]
PeriodicSite: Na (4.44, 2.563, 10.63) [0.6667, 0.3333, 0.6667]
PeriodicSite: Na (2.96, 5.127, 10.63) [0.6667, 0.6667, 0.6667]
PeriodicSite: Na (5.92, 8.545e-11, 10.63) [0.6667, 1.111e-11, 0.6667]
PeriodicSite: Na (7.4, 2.563, 10.63) [1.0, 0.3333, 0.6667]
PeriodicSite: Na (5.92, 5.127, 10.63) [1.0, 0.6667, 0.6667]
PeriodicSite: Na (8.88, 8.545e-11, 10.63) [1.0, 1.111e-11, 0.6667]
PeriodicSite: Na (2.96, 1.709, 3.625e-16) [0.4444, 0.2222, 0.0]
PeriodicSite: Na (1.48, 4.272, 5.437e-16) [0.44

## 셀 4 — Stage 1: 조성 열거

27개 TM 자리의 **정확 정수 전하식** `Σ n_i·v_i = +81`. 다가 원소는 `DEFAULT_STATE`에서 출발하고 부족/과잉 전하는 호스트 redox 쌍이 먼저 흡수합니다.
채택된 배분(`n_Ni3`, `n_Mn3`, `n_Co2`, `n_V3`, `n_V5`)이 **DFT 자화 감사가 검증할 예측**입니다.

격자 A(Ni6Mn6 + 3-3-3-3-3) · B(Ni4Mn4 + 4-4-4-4-3) · C(구제 격자, A도 B도 정수해가 없는 조합에만).
`compositions_master.csv`는 **입력 전용**이라 이미 있으면 md5가 같은지만 확인하고 절대 덮어쓰지 않습니다.

In [8]:
# ============ CELL 4: STAGE 1 — enumeration ============
import pandas as pd, hashlib, json
fp = f"{WORKDIR}/results/compositions_master.csv"
comps, n_legacy = W.enumerate_compositions(use_exact=True)
COMP_HASH = hashlib.md5(comps.to_csv(index=False).encode()).hexdigest()

if os.path.exists(fp):
    on_disk = hashlib.md5(open(fp, "rb").read()).hexdigest()
    assert on_disk == COMP_HASH, ("compositions_master.csv on disk differs from this enumeration "
                                  f"({on_disk} != {COMP_HASH}) -> delete it or restore the CONFIG")
    print("existing master reused, md5 identical:", len(comps), "rows")
else:
    tmp = fp + ".tmp"; comps.to_csv(tmp, index=False); os.replace(tmp, fp)   # atomic
    open(f"{WORKDIR}/results/compositions.md5", "w").write(COMP_HASH)
    print("master written:", len(comps), "rows")

print("exact-neutral:", len(comps), comps.scheme.value_counts().to_dict())
print("legacy-tolerance count (v1 reproduction, expect 7279):", n_legacy)
ent = {s: round(1000*W.K_B*W.T_SYNTH*comps[comps.scheme == s].sconf_per_atom_kB.mean(), 1) for s in "ABC"}
print("entropy bonus at T_synth, A/B/C (meV/atom):", ent)
print("compositions md5:", COMP_HASH)
# ---- invariant 3: the scheme-A entropy term must land in 42-52 meV/atom
assert 42 <= ent["A"] <= 52, f"entropy term {ent['A']} meV/atom out of range -> check the 27/108 factor or the log"
comps.head(3)

existing master reused, md5 identical: 7073 rows
exact-neutral: 7073 {'A': 3773, 'B': 2982, 'C': 318}
legacy-tolerance count (v1 reproduction, expect 7279): 7279
entropy bonus at T_synth, A/B/C (meV/atom): {'A': np.float64(46.0), 'B': np.float64(47.5), 'C': np.float64(46.4)}
compositions md5: 0fc86c95929b0c775d7ed0505cafc76a


,comp_id,scheme,dopants,B_short_dopant,C_short_dopant,C_long_dopant,site_counts,n_species,sconf_kB,sconf_per_atom_kB,n_Ni3,n_Mn3,n_Co2,n_V3,n_V5,formula_cell
0,A_Al-Cu-Co-Ca-V,A,Al-Cu-Co-Ca-V,,,,"{""Ni"": 6, ""Mn"": 6, ""Al"": 3, ""Cu"": 3, ""Co"": 3, ...",7,1.8892,0.4723,3,0,0,0,0,Na27Ni6Mn6Al3Cu3Co3Ca3V3O54
1,A_Al-Cu-Ga-Ca-V,A,Al-Cu-Ga-Ca-V,,,,"{""Ni"": 6, ""Mn"": 6, ""Al"": 3, ""Cu"": 3, ""Ga"": 3, ...",7,1.8892,0.4723,3,0,0,0,0,Na27Ni6Mn6Al3Cu3Ga3Ca3V3O54
2,A_Al-Cu-Ga-Co-Ca,A,Al-Cu-Ga-Co-Ca,,,,"{""Ni"": 6, ""Mn"": 6, ""Al"": 3, ""Cu"": 3, ""Ga"": 3, ...",7,1.8892,0.4723,6,0,0,0,0,Na27Ni6Mn6Al3Cu3Ga3Co3Ca3O54


## 셀 5 — ★ 앵커 게이트 (재기 전에 자부터)

답을 아는 두 계로 자의 방향을 검증합니다 (2026-09-03 재설계, A안 = 상대 게이트):

- **HOST** NaNi₀.₅Mn₀.₅O₂ → 탈소듐 시 **P3로 gliding** (Komaba 2012). 13.5/13.5는 27자리에 못 앉으므로 두 변형 평균 — 1차 전하 오차가 상쇄됩니다.
- **HEO** NaNi₀.₁₂Cu₀.₁₂Mg₀.₁₂Fe₀.₁₅Co₀.₁₅Mn₀.₁Ti₀.₁Sn₀.₁Sb₀.₀₄O₂ (Zhao et al., *Angew. Chem.* **59**, 264 (2020)) → O3→P3 전이가 **지연**됨(용량의 60% 이상이 O3 영역). 반정수 문제가 없어 두 변형은 같은 조성에 다른 난수(SQS·빈자리) — 9원소 배열 노이즈를 평균합니다.

게이트는 **절대 상 판정이 아니라 자의 방향**을 요구합니다: `dE_desod(HEO) > dE_desod(HOST) + margin`.
x=0.52에서 HEO가 실험적으로 O3라는 보장이 없기 때문입니다(전이가 늦을 뿐 사라지지 않음). 스크리닝의 목표가
"HEO보다 더 버티는 조성"이므로, 앵커는 눈금 방향만 보정하고 억제 판정(`phase_desod==O3`)은 후보의 목표 정의로 남습니다.

> **Ti03 앵커 삭제(2026-09-03).** "Ti₀.₃이 x=0.52에서 O3 유지"는 출처가 없었고, 1차 문헌(Wang et al., *Adv. Mater.* **29**,
> 1700210 (2017))은 NaNi₀.₅Mn₀.₅₋ₓTiₓO₂ 전 계열에서 **가역적 O3−P3 전이**를 보고합니다. 실제 ORB 실행(2026-09-02)도
> Ti03 → P3 (dE_desod −31)를 냈는데, 이는 모델 오류가 아니라 문헌과 일치하는 답이었습니다.

전압 창은 [2.5, 3.4]로 완화 — GGA 계열 화폐는 층상 산화물 전압을 0.3–0.5 V 과소평가합니다(Aydinol PRB 1997). 실측 2.685 V는 실험 3.1 V 대비 정확히 그 오프셋.

앵커를 워커가 아니라 여기서 한 번만 돌리는 이유: 샤드마다 중복 계산하면 같은 `anchors.csv`와 같은 CIF에 동시에 쓰는 경쟁이 생깁니다.

`sigma_paired`와 `sigma_max`를 나란히 찍습니다. **`sigma_paired < sigma_max`이면 짝지은 빈자리가 실제로 오차를 상쇄하고 있다는 증거**입니다.
(mock에서는 빈자리 벌점이 O3/P3 독립 난수라 반대로 나옵니다 — 실전에서만 판별됩니다.)

**게이트 지점 이동(2026-09-03, 승인)**: Zhao 2020의 만충전이 x≈0.52(110 mAh/g ≈ Δx 0.48)라 그 지점에선 실험도 HEO가 P3 —
방향 게이트를 실험이 두 앵커를 실제로 구분하는 **x = 20/27 ≈ 0.74**(Na 7개 제거, `{ph}_mild_{k}` 태그)로 옮김.
x=0.52 gap은 기록만 하고 게이트하지 않음. 절대 상 판정은 어느 x에서도 요구하지 않음(방향형 유지 — 계통 오차 상쇄).

**게이트 v4 (2026-09-03, 승인)**: ORB(−4.9)·MACE(−5.2)가 x=0.74 gap을 1 meV 안쪽으로 일치 판독 → 탈소듐 순서는
0 K PBE 열역학에 없는 신호(전이 지연 = 엔트로피/운동학 기원, npj Comput. Mater. 2025)로 판정하고 **진단으로 강등**.
게이트는 화폐가 실제로 분해하는 **pristine 방향**(`dE_pris(HEO) − dE_pris(HOST) > margin`, ORB +20.0/MACE +23.4)으로 이동,
"자가 충실한가"의 검증은 다음 셀 5c(단일 TM 문헌 DFT 캘리브레이션)가 담당.


In [9]:
# ============ CELL 5: ANCHOR GATE ============
# Gate history (all user-approved, 2026-09-03):
#   v1  absolute Ti03 anchor -- removed (no literature support; Wang Adv. Mater. 2017 contradicts).
#   v2  directional gate at x=0.52 -- retired: Zhao 2020 fully charges the HEO to x~0.52
#       (110 mAh/g of ~229 mAh/g per Na), so BOTH anchors are P3 there in experiment.
#   v3  directional gate at x=0.74 -- retired as a GATE after ORB (-4.9) and MACE (-5.2)
#       agreed within 1 meV/f.u. on a ~0 gap: two independent architectures on one training
#       currency read the same value, so the missing signal is not model noise -- the O3->P3
#       delay in the HEO is not contained in 0 K PBE-family thermodynamics (entropic/kinetic
#       origin; cf. the npj Comput. Mater. 2025 entropic-control study, doi:10.1038/s41524-025-01954-2).
#       gap_074/gap_052 stay RECORDED as diagnostics: they are paper content, not gates.
#   v4  (current) directional gate on the PRISTINE preference, where the currency does resolve
#       the two anchors (ORB +20.0, MACE +23.4 meV/f.u.) -- plus the single-TM literature-DFT
#       calibration in CELL 5c, which takes over the "is the ruler faithful?" role.
import numpy as np, pandas as pd, json, time
ckpt_main = f"{WORKDIR}/checkpoints/energies_main.csv"
done = W.load_done_all()
W.merged_energies()          # invariant 6: one currency in the checkpoints, checked BEFORE any GPU time

E_NA = W.na_reference(ckpt_main, done)
print("E(Na)/atom =", round(E_NA, 4), "eV   (PBE reference ~ -1.31)")
json.dump({"E_Na_per_atom": E_NA, "model": W.MODEL_TAG}, open(f"{WORKDIR}/results/E_Na.json", "w"))

t0 = time.time()
items = [it for aid, c in W.ANCHORS.items() for w in ("all", "mild") for it in W.build_items(aid, c, w)]
print(f"{len(items)} anchor structures built in {time.time()-t0:.1f}s  (incl. x=0.74 diagnostic point)")
W.relax_batch(items, ckpt_main, done, counts_of=W.ANCHORS, save_cif=True)

X_MILD = (W.N_NA - W.N_NA_REMOVE_MILD) / W.N_NA          # 20/27 = 0.741

def mild_stats(recs):
    """Paired minimum and spread at x=0.74, same conventions as metrics_from (section 3):
    min_k(E_P3,k - E_O3,k) in meV/f.u., O3 and P3 sharing each vacancy pattern k."""
    ks = [k for k in range(W.N_VAC_SAMPLES) if f"O3_mild_{k}" in recs and f"P3_mild_{k}" in recs]
    assert ks, "no mild-desodiation records -> worker missing the 'mild' build path?"
    d = [(recs[f"P3_mild_{k}"]["energy"] - recs[f"O3_mild_{k}"]["energy"]) * 1000 / W.N_FU for k in ks]
    return float(min(d)), float(np.std(d))

R = W.recs_by_comp(W.merged_energies(), list(W.ANCHORS))
rows = []
for aid in W.ANCHORS:
    m = W.metrics_from(R[aid]); m["anchor"] = aid; m["group"] = aid.split("_")[0]
    o3 = [R[aid][f"O3_desod_{k}"]["energy"] for k in range(W.N_VAC_SAMPLES)]
    p3 = [R[aid][f"P3_desod_{k}"]["energy"] for k in range(W.N_VAC_SAMPLES)]
    m["sigma_max"]      = max(np.std(o3), np.std(p3))*1000/W.N_FU   # the unpaired estimate, for comparison
    m["dE_desod_indep"] = (min(p3) - min(o3))*1000/W.N_FU           # the pre-pairing definition
    m["dE_mild"], m["sigma_mild"] = mild_stats(R[aid])
    rows.append(m)
A = pd.DataFrame(rows)
cols = ["V_avg","dE_pris","dE_desod","dE_mild","ddE","dV_O3","dV_trans",
        "sigma_vac","sigma_mild","sigma_max","dE_desod_indep"]
Ag = A.groupby("group")[cols].mean()
Ag["delta"]      = A.groupby("group").sigma_vac.max().clip(lower=W.DELTA_FLOOR_MEV_FU)
Ag["delta_mild"] = A.groupby("group").sigma_mild.max().clip(lower=W.DELTA_FLOOR_MEV_FU)
Ag["phase_desod"] = [W.judge_phase(r.dE_desod, r.delta) for r in Ag.itertuples()]
Ag["phase_mild"]  = [W.judge_phase(r.dE_mild, r.delta_mild) for r in Ag.itertuples()]
Ag["variant_sign_agree"] = A.groupby("group").dE_pris.apply(lambda s: len(set(np.sign(s))) == 1)
A.to_csv(f"{WORKDIR}/results/anchors_raw.csv", index=False)
Ag.to_csv(f"{WORKDIR}/results/anchors.csv")
print(Ag.round(3).to_string())
print("\npaired-vacancy check:  sigma_paired", Ag.sigma_vac.round(2).to_dict(),
      "vs sigma_max", Ag.sigma_max.round(2).to_dict())
print("  paired < max -> the shared vacancy pattern is cancelling, as intended.")
print("  paired > max -> O3 and P3 vacancy energies are uncorrelated; pairing buys nothing, tell Claude.")

# ---- desodiated orderings: DIAGNOSTICS ONLY (kept for the paper, never gated) ----
gap_074 = float(Ag.loc["HEO", "dE_mild"] - Ag.loc["HOST", "dE_mild"])
gap_052 = float(Ag.loc["HEO", "dE_desod"] - Ag.loc["HOST", "dE_desod"])
print(f"\n[diagnostic] gap @ x={X_MILD:.2f} = {gap_074:+.1f}  |  gap @ x=0.52 = {gap_052:+.1f} meV/f.u.")
print("  ~0 or negative here is the EXPECTED stock-uMLIP reading (ORB -4.9 / MACE -5.2 on"
      "\n  2026-09-03): the experimental O3->P3 delay is not a 0 K-thermodynamic signal.")

# ---- invariant 5 (v4): gate on what 0 K thermodynamics does support ----
# (a) sanity: the HOST voltage over x=1 -> 0.52.  [2.5, 3.4] allows the systematic 0.3-0.5 V
#     GGA-currency underestimate (Aydinol PRB 1997); experiment ~3.1 V, real ORB 2.69 V.
assert 2.5 <= Ag.loc["HOST", "V_avg"] <= 3.4, f"HOST voltage {Ag.loc['HOST','V_avg']:.2f} V outside [2.5, 3.4]"
# (b) the negative anchor: HOST must glide at full charge, as in experiment (Komaba 2012).
assert Ag.loc["HOST", "phase_desod"] == "P3", "HOST must glide (P3) at x=0.52 -> model or template problem"
# (c) the pristine DIRECTION: at x=1 the currency resolves the chemistry (both uMLIPs read
#     HEO's O3 preference ~20 meV/f.u. above HOST's).  margin = the variant spread of the
#     2-variant mean (std over 2 values = half their difference), floored at 5 meV/f.u.
gap_pris    = float(Ag.loc["HEO", "dE_pris"] - Ag.loc["HOST", "dE_pris"])
margin_pris = float(max(A.groupby("group").dE_pris.std(ddof=0).max(), W.DELTA_FLOOR_MEV_FU))
print(f"\nruler direction @ x=1 (pristine): dE_pris(HEO) - dE_pris(HOST) = {gap_pris:+.1f} meV/f.u."
      f"  (margin {margin_pris:.1f})")
assert gap_pris > margin_pris, \
    f"pristine ruler direction wrong or lost in noise: gap {gap_pris:+.1f} <= margin {margin_pris:.1f} meV/f.u."

RULER = dict(dE_desod_HOST=float(Ag.loc["HOST", "dE_desod"]), dE_desod_HEO=float(Ag.loc["HEO", "dE_desod"]),
             ddE_HOST=float(Ag.loc["HOST", "ddE"]), ddE_HEO=float(Ag.loc["HEO", "ddE"]),
             dE_pris_HOST=float(Ag.loc["HOST", "dE_pris"]), dE_pris_HEO=float(Ag.loc["HEO", "dE_pris"]),
             dE_mild_HOST=float(Ag.loc["HOST", "dE_mild"]), dE_mild_HEO=float(Ag.loc["HEO", "dE_mild"]),
             x_mild=X_MILD, gap=gap_pris, margin=margin_pris, gap_074=gap_074, gap_052=gap_052,
             E_Na=float(E_NA), model=W.MODEL_TAG)
json.dump(RULER, open(f"{WORKDIR}/results/ruler.json", "w"), indent=1)
print("\nRULER (meV/f.u.):", {k: round(v, 1) for k, v in RULER.items() if isinstance(v, float)})
if gap_052 <= 0:
    print("[note] gap @ x=0.52 <= 0: cell 11's ruler_pos (built on the x=0.52 anchors) has an inverted"
          "\n       denominator -- read it as a coordinate, not as 'closer to 1 = better'.")
print("ANCHOR GATE PASSED -> run the CELL 5c calibration next")


E(Na)/atom = -1.3046 eV   (PBE reference ~ -1.31)
88 anchor structures built in 0.2s  (incl. x=0.74 diagnostic point)
       V_avg  dE_pris  dE_desod  dE_mild      ddE  dV_O3  dV_trans  sigma_vac  sigma_mild  sigma_max  dE_desod_indep   delta  delta_mild phase_desod phase_mild  variant_sign_agree
group                                                                                                                                                                              
HEO    2.723  138.850   -31.342   52.876 -170.192  2.607     1.237     11.501       8.078     10.841         -17.681  15.026      12.922          P3         O3                True
HOST   2.690  119.372   -14.306   51.597 -133.678  4.417     1.838      3.097       4.945      3.007         -12.707   5.000       5.035          P3         O3                True

paired-vacancy check:  sigma_paired {'HEO': 11.5, 'HOST': 3.1} vs sigma_max {'HEO': 10.84, 'HOST': 3.01}
  paired < max -> the shared vacancy pattern is cancelli

## 셀 5c — ★ 자 캘리브레이션 (단일 TM, 문헌 0 K DFT 대조)

스크리닝이 의존하는 사슬은 `MLIP →(1)→ PBE 화폐 →(2)→ 실험` 중 **(1)뿐**입니다(같은 화폐 안에서의 랭킹).
(2)의 탈소듐 구간은 엔트로피/운동학이라 top-30 DFT·유한온도 후속 연구 몫. 이 셀은 (1)을 **발표된 0 K DFT가 있는
단순계**로 검증합니다: NaCoO₂/NaNiO₂/NaMnO₂ (27×3가 = +81 정확, SQS 불필요, 몇 분 계산).
assert는 확실한 문헌 사실만(부호·단조성·Co의 x=0.52 P3): Toumar PR Applied 2015, Kaufman & Van der Ven PRM 2019,
Delmas 1981 (O3→O′3→P3). 표의 정량값(기울기/연화)은 논문 그림과 눈으로 대조 — 실패 시 스크리닝 중단.

In [10]:
# ============ CELL 5c: RULER CALIBRATION — single-TM set vs literature 0 K DFT ============
# The screening only needs link (1) of the chain  MLIP --(1)--> PBE-family DFT --(2)--> experiment:
# it ranks 7,073 compositions inside ONE currency.  Link (2) at desodiated x is entropic/kinetic
# (cell 5 diagnostics) and belongs to the top-30 DFT + finite-T follow-up.  This cell certifies
# link (1) on chemistries where the 0 K DFT answer is PUBLISHED, so no new DFT is needed:
#   NaCoO2 / NaNiO2 / NaMnO2 -- 27 x M(3+) = +81 satisfies charge neutrality exactly, single
#   element needs no SQS, and the O3/P3 x-dependence is established GGA(+U) literature
#   (Toumar et al. Phys. Rev. Applied 4, 064002 (2015): O3-type NaxMO2 voltages/orderings;
#    Kaufman & Van der Ven PRM 3, 015402 (2019): NaxCoO2 stacking hulls; experimental sequence
#    O3 -> O'3 -> P3 on charge: Delmas et al. 1981).
# Asserted here are only the ROBUST literature facts (signs and orderings); the printed table is
# for quantitative comparison against the papers' figures (slope/softening), not asserted.
import numpy as np, pandas as pd, json

CAL = {"CAL_Co": {"Co": 27}, "CAL_Ni": {"Ni": 27}, "CAL_Mn": {"Mn": 27}}
assert np.isfinite(W.E_NA_GLOBAL[0]), "E_Na not set -> run cell 5 first"

def cal_items(cid, counts):
    o3 = W.place_tm_random(counts, W.comp_rng(cid))       # single element: placement is trivial
    p3 = W.o3_to_p3(o3, glide=W.GLIDE)
    yield cid, "O3_pris", o3
    yield cid, "P3_pris", p3
    for salt0, nrm, name in ((100, W.N_NA_REMOVE, "desod"), (200, W.N_NA_REMOVE_MILD, "mild")):
        for k in range(W.N_VAC_SAMPLES):
            pat = W.vacancy_pattern(o3, W.comp_rng(cid, salt=salt0+k), n_remove=nrm)
            for ph, st in (("O3", o3), ("P3", p3)):
                d = st.copy(); d.remove_sites(pat)
                yield cid, f"{ph}_{name}_{k}", d

items = [it for cid, c in CAL.items() for it in cal_items(cid, c)]
print(f"{len(items)} calibration structures (3 compounds x [2 pris + 2x{W.N_VAC_SAMPLES} desod"
      f" + 2x{W.N_VAC_SAMPLES} mild])")
W.relax_batch(items, ckpt_main, done, counts_of=CAL, save_cif=True)

def paired_min(recs, name):
    ks = [k for k in range(W.N_VAC_SAMPLES) if f"O3_{name}_{k}" in recs and f"P3_{name}_{k}" in recs]
    d  = [(recs[f"P3_{name}_{k}"]["energy"] - recs[f"O3_{name}_{k}"]["energy"]) * 1000 / W.N_FU for k in ks]
    return float(min(d)), float(np.std(d))

Rc = W.recs_by_comp(W.merged_energies(), list(CAL))
crow = []
for cid in CAL:
    m = W.metrics_from(Rc[cid])                                   # dE_pris, dE_desod, V_avg (x=0.52 window)
    m["compound"] = cid.replace("CAL_", "Na") + "O2"
    m["dE_mild"], m["sigma_mild"] = paired_min(Rc[cid], "mild")
    # crossing x where dE(P3-O3) = 0, linear between the bracketing pair of the 3 computed x points
    xs = [1.0, (W.N_NA - W.N_NA_REMOVE_MILD)/W.N_NA, (W.N_NA - W.N_NA_REMOVE)/W.N_NA]
    ds = [m["dE_pris"], m["dE_mild"], m["dE_desod"]]
    m["x_cross"] = float("nan")
    for (x1, d1), (x2, d2) in zip(zip(xs, ds), list(zip(xs, ds))[1:]):
        if d1 > 0 >= d2:
            m["x_cross"] = x1 + (x2 - x1) * d1 / (d1 - d2); break
    crow.append(m)
C = pd.DataFrame(crow).set_index("compound")
show = ["V_avg", "dE_pris", "dE_mild", "dE_desod", "sigma_vac", "sigma_mild", "x_cross"]
print(C[show].round(3).to_string())
C.to_csv(f"{WORKDIR}/results/calibration.csv")
json.dump({"model": W.MODEL_TAG, "table": C[show].round(3).to_dict()},
          open(f"{WORKDIR}/results/calibration.json", "w"), indent=1, default=str)

# ---- calibration invariants: the ruler must reproduce the published 0 K facts ----
# (1) at x=1 every NaMO2 prefers O-type stacking (synthesis + every published hull)
assert (C.dE_pris > 0).all(), f"calibration: O3 not preferred at x=1 -> broken ruler\n{C.dE_pris}"
# (2) desodiation must monotonically favor P-type: dE_pris > dE_mild > dE_desod
assert ((C.dE_pris > C.dE_mild) & (C.dE_mild > C.dE_desod)).all(), \
    f"calibration: dE(x) not monotonic -> broken ruler\n{C[['dE_pris','dE_mild','dE_desod']]}"
# (3) the cleanest case, NaxCoO2: P-type is favored by mid-desodiation (O3->O'3->P3, Delmas 1981;
#     the published stacking hulls agree) -> dE at x=0.52 must be negative
assert C.loc["NaCoO2", "dE_desod"] < 0, \
    f"calibration: NaxCoO2 not P3 at x=0.52 (dE={C.loc['NaCoO2','dE_desod']:+.1f}) -> broken ruler"
# (4) unit sanity (invariant 2 applied to the calibration set)
assert C[["dE_pris", "dE_mild", "dE_desod"]].abs().max().max() < 500, "calibration: unit slip"
# soft check, not asserted: GGA-family average voltages of these compounds sit around 2-3.3 V
for cmpd, v in C.V_avg.items():
    if not (1.8 <= v <= 3.6):
        print(f"  [warn] {cmpd} V_avg {v:.2f} V outside the loose GGA window [1.8, 3.6] -- compare"
              f" against Toumar 2015 before trusting the run")
print("\nCALIBRATION PASSED -> the ruler reproduces the published 0 K stacking facts;"
      "\ncompare the table against Toumar 2015 / Kaufman 2019 figures for the slope, then launch PASS 1")


66 calibration structures (3 compounds x [2 pris + 2x5 desod + 2x5 mild])
          V_avg  dE_pris  dE_mild  dE_desod  sigma_vac  sigma_mild  x_cross
compound                                                                   
NaCoO2    2.088   75.858   33.802   -40.604      3.563       2.813    0.640
NaNiO2    2.668  112.309   24.424   -17.166      4.603      10.019    0.610
NaMnO2    2.073  142.101   53.783   -18.846      9.696      10.495    0.576

CALIBRATION PASSED -> the ruler reproduces the published 0 K stacking facts;
compare the table against Toumar 2015 / Kaufman 2019 figures for the slope, then launch PASS 1


## 셀 6 — PASS 1 런처 (pristine O3/P3 전수)

GPU당 워커 하나, 각자 자기 체크포인트 CSV에만 씁니다(경쟁 없음). 이 셀은 **즉시 반환**하니 셀 7로 지켜보세요.
워커는 100조성씩 끊어 **SQS를 전체 CPU 코어로 먼저 생성한 뒤** GPU 배치로 넘깁니다 — GPU가 SQS를 기다리며 놀지 않게 하는 핵심입니다.

In [14]:
# ============ CELL 6: PASS 1 LAUNCHER ============
import subprocess, sys, os
PROCS = {}
def launch(pass_no):
    total = N_PODS * N_GPUS; procs = []
    for k in range(N_GPUS):
        wid = POD_ID * N_GPUS + k
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
        log = open(f"{WORKDIR}/checkpoints/worker_{wid}_p{pass_no}.log", "a")
        cmd = [sys.executable, "heo_worker.py", "--worker-id", str(wid),
               "--n-workers", str(total), "--pass-no", str(pass_no)]
        if LIMIT: cmd += ["--limit", str(LIMIT)]
        p = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
        procs.append(p); print(f"worker {wid} -> GPU {k} (pid {p.pid})")
    PROCS[pass_no] = procs
    print(f"\n{len(procs)} workers running (PASS {pass_no}). This cell returns now - watch with the next cell.")
    return procs

launch(1)

worker 0 -> GPU 0 (pid 751)

1 workers running (PASS 1). This cell returns now - watch with the next cell.


[<Popen: returncode: None args: ['/usr/local/bin/python', 'heo_worker.py', '-...>]

## 셀 7 — 모니터 (PASS 1이 끝날 때까지 반복 실행)

In [15]:
# ============ CELL 7: MONITOR ============
# Re-run this cell to refresh, or call monitor(1, wait=True) to block until the pass finishes.
import glob, time, subprocess
import pandas as pd

def _gpu_line():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used",
                               "--format=csv,noheader"], capture_output=True, text=True).stdout
    except (FileNotFoundError, OSError):
        return "(no nvidia-smi)\n"

def monitor(pass_no, wait=False, every=120):
    while True:
        for lf in sorted(glob.glob(f"{WORKDIR}/checkpoints/worker_*_p{pass_no}.log")):
            lines = [l for l in open(lf).read().strip().split("\n") if l.strip()]
            print(f"--- {lf.split('/')[-1]} ---"); print("\n".join(lines[-3:]) or "  (no output yet)")
        procs = PROCS.get(pass_no, [])
        codes = [p.poll() for p in procs]; alive = sum(c is None for c in codes)
        print(f"\nalive: {alive}/{len(procs)} | exit codes: {codes}")
        fps = glob.glob(f"{WORKDIR}/checkpoints/energies*.csv")
        if fps: print("relaxed structures on disk:", sum(len(pd.read_csv(f)) for f in fps))
        print(_gpu_line())
        if procs and alive == 0:
            # a non-zero exit means the shard stopped early; its compositions are simply missing
            # downstream, so surface it here instead of letting cell 8 aggregate a partial pass.
            assert all(c == 0 for c in codes), \
                f"a worker exited non-zero {codes} -> read its log, fix, re-run the launcher (checkpoints resume)"
            print(f"PASS {pass_no} COMPLETE -> continue to the next cell")
            return
        if not wait: return
        time.sleep(every); print("\n" + "-"*60)

monitor(1)

--- worker_0_p1.log ---
[w0] DONE in 0.00 h
[w0] PASS1 shard: 0 comps (model orb_v3_conservative_inf_mpa, engine torch-sim)
[w0] DONE in 0.00 h
--- worker_1_p1.log ---
[w1] DONE in 0.13 h
[w1] PASS1 shard: 0 comps (model orb_v3_conservative_inf_mpa, engine torch-sim)
[w1] DONE in 0.00 h
--- worker_2_p1.log ---
[w2] DONE in 0.19 h
[w2] PASS1 shard: 0 comps (model orb_v3_conservative_inf_mpa, engine torch-sim)
[w2] DONE in 0.00 h
--- worker_3_p1.log ---
[w3] DONE in 0.89 h
[w3] PASS1 shard: 0 comps (model orb_v3_conservative_inf_mpa, engine torch-sim)
[w3] DONE in 0.00 h
--- worker_4_p1.log ---
Model Memory Estimation: Running forward pass on state with 9288 atoms and 86 systems.
[w4] 1179/1179 | 1303 comps/h | ETA 0.0 h
[w4] DONE in 0.90 h
--- worker_5_p1.log ---
Model Memory Estimation: Running forward pass on state with 9288 atoms and 86 systems.
[w5] 1178/1178 | 1507 comps/h | ETA 0.0 h
[w5] DONE in 0.78 h

alive: 0/1 | exit codes: [0]
relaxed structures on disk: 53960
0, 0 %, 4 MiB


## 셀 7b — ★ 배리어: PASS 1 완료까지 대기 (Run All 안전장치)

## 셀 8 — Stage 2: hull (축 ①)

MP에서 **조성 원소 풀의 부분집합인 안정상**을 한 번에 받아 **같은 모델로 재이완**한 뒤 `pymatgen.PhaseDiagram`으로 `Ehull`을 구합니다.

v1이 여기서 한 번 죽었습니다 — 1원자짜리 원소 셀이 배치에 섞이니 autobatcher가 배치를 25만 원자까지 키우려다 폭주했습니다.
그래서 **24원자 미만은 슈퍼셀로 확장하고 크기순으로 정렬해** 배치를 균질하게 만듭니다(v1 CELL 7-FIX). 확장한 만큼 조성도 `factor`로 스케일합니다.

`Ehull_eff = Ehull − k_B·T·S_conf·(27/108)`, `T* = Ehull / (k_B·S_conf·27/108)`.

In [16]:
# ============ CELL 8: HULL + PASS-1 AGGREGATION ============
import math, json, time, os
import numpy as np, pandas as pd
from pathlib import Path
from scipy.optimize import linprog
from pymatgen.core import Structure, Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry

CACHE = f"{WORKDIR}/hull_cache/competitors.json"
ckpt_main = f"{WORKDIR}/checkpoints/energies_main.csv"
HEARTBEAT = Path(WORKDIR, "results", ".heartbeat")   # touched inside long CPU loops so the idle watchdog sees activity
done = W.load_done_all()
comps = pd.read_csv(f"{WORKDIR}/results/compositions_master.csv")
if LIMIT: comps = comps.sample(LIMIT, random_state=W.SEED).sort_values("comp_id").reset_index(drop=True)

# ---- mock: no MP, no relaxation; the synthetic Ehull keeps the downstream chain testable ----
MOCK_HULL = (MODE == "mock")
if MOCK_HULL: print("MODE=mock -> synthetic Ehull, MP is not queried")

# ---- competing phases: stable entries whose elements are a subset of our pool ----
# Section-6 item 4, verified 2026-09-03: exclude_elements is capped at 60 CHARACTERS server-side
# (HTTP 422), so one big exclusion query cannot work.  Inverted two-step instead:
#   (1) one LIGHT query listing every stable entry (id + element list only, ~40k rows, no structures),
#   (2) filter locally to element sets inside our pool, (3) fetch structures for just those ids in chunks.
if not MOCK_HULL and not os.path.exists(CACHE):
    assert MP_API_KEY, "export MP_API_KEY before running this cell"
    from mp_api.client import MPRester
    pool = sorted({e for sc in comps.site_counts for e in json.loads(sc)} | {"Na", "O"})
    pset = set(pool)
    with MPRester(MP_API_KEY) as mpr:
        light = mpr.materials.summary.search(is_stable=True, num_elements=(1, 7),
                                             fields=["material_id", "elements"])
        keep = [str(d.material_id) for d in light if {str(el) for el in d.elements} <= pset]
        print(f"stable MP entries: {len(light)} -> inside our {len(pool)}-element pool: {len(keep)}")
        raw = {}
        for i0 in range(0, len(keep), 200):
            for d in mpr.materials.summary.search(material_ids=keep[i0:i0+200],
                                                  fields=["material_id", "structure"]):
                raw[str(d.material_id)] = d.structure.as_dict()
    assert len(raw) > 200, f"only {len(raw)} competing phases -> the query looks wrong; do not build a hull on this"
    json.dump({"model": W.MODEL_TAG, "entries": raw}, open(CACHE, "w"))
    print("MP query returned", len(raw), "stable phases")
if MOCK_HULL:
    pool_struct = {}
else:
    _c = json.load(open(CACHE))
    # The hull currency must be the screening currency. The cache carries its model tag, so a leftover
    # file from another model can never be silently reused (invariant 6 on the hull side).
    assert _c.get("model") == W.MODEL_TAG, f"hull cache built with {_c.get('model')} != {W.MODEL_TAG} -> delete {CACHE}"
    pool_struct = _c["entries"]
    print("competing phases:", len(pool_struct))

# ---- re-relax them with the SAME model ----
# v1 died here: 1-atom elemental cells in the batch made the autobatcher try 255,066 atoms per batch.
# Expanding small cells to >=24 atoms and sorting by size keeps the batches homogeneous (v1 CELL 7-FIX).
items = []
for mid, sd in pool_struct.items():
    st = Structure.from_dict(sd)
    if len(st) < 24:
        n = math.ceil((24 / len(st)) ** (1/3)); st = st * (n, n, n)
    items.append(("HULL", mid, st))
items.sort(key=lambda x: len(x[2]))
t0 = time.time()
for i in range(0, len(items), 200):
    W.relax_batch(items[i:i+200], ckpt_main, done, save_cif=False)
    print(f"HULL {min(i+200, len(items))}/{len(items)} | {(time.time()-t0)/60:.1f} min", flush=True)

allE = W.merged_energies()
hdf = allE[allE.comp_id == "HULL"].set_index("tag")
entries = []
for mid, sd in pool_struct.items():
    if mid not in hdf.index: continue
    st = Structure.from_dict(sd)
    factor = float(hdf.loc[mid, "natoms"]) / len(st)      # the energy is for the EXPANDED cell
    entries.append(PDEntry(st.composition * factor, float(hdf.loc[mid, "energy"]), name=mid))
print("hull entries:", len(entries), "| skipped:", len(pool_struct) - len(entries))

# ---- Ehull, Ehull_eff, T* for every pristine O3 cell ----
R1 = W.recs_by_comp(allE[~allE.comp_id.isin(["REF", "HULL"])])
# Coverage guard: cell 7b should have blocked until PASS 1 finished; if cells were run out of
# order this catches the half-filled checkpoint instead of silently skipping compositions.
# (98%, not 100%: relax_batch drops individual structures that fail their own relaxation.)
_have = sum(1 for r in comps.itertuples() if {"O3_pris", "P3_pris"} <= set(R1.get(r.comp_id, {})))
assert _have >= 0.98 * len(comps), \
    f"PASS 1 incomplete: {_have}/{len(comps)} compositions have both pristine cells -> wait (cell 7b)"
print(f"pass-1 coverage: {_have}/{len(comps)}")

# 2026-09-04: the per-composition PhaseDiagram was the bottleneck (hours, and it killed the kernel).
# Every competitor is an MP *stable* phase, i.e. a hull vertex, so qhull's 8-D hull of ~200 vertices is
# the worst case for facet count and memory.  We never need the hull itself, only the hull energy at ONE
# composition, and that is a small linear programme:
#     E_hull(c) = min_x sum_i x_i e_i   s.t.  sum_i x_i f_i = f_c,  x_i >= 0
# (x_i = atom fraction taken from entry i, e_i = eV/atom, f = atomic-fraction vectors).  This is exactly
# what get_e_above_hull evaluates (checked to 1e-15 eV/atom on synthetic data; re-checked below on the
# first HULL_XCHECK real compositions).  Ehull = max(0, e_per_atom - E_hull), i.e. pymatgen semantics.
ENTRY_FRAC = [{str(el): en.composition.get_atomic_fraction(el) for el in en.composition.elements} for en in entries]
ENTRY_EPA  = np.array([en.energy_per_atom for en in entries])
_sub_cache = {}         # element set -> (entry indices, A matrix, element order); ~4,400 distinct sets over 7,073 rows

def hull_energy_per_atom(comp, key):
    if key not in _sub_cache:
        els = sorted(key)
        idx = [i for i, f in enumerate(ENTRY_FRAC) if set(f) <= key]
        A = np.array([[ENTRY_FRAC[i].get(el, 0.0) for i in idx] for el in els])
        _sub_cache[key] = (idx, A, els)
    idx, A, els = _sub_cache[key]
    b = np.array([comp.get_atomic_fraction(el) for el in els])
    res = linprog(ENTRY_EPA[idx], A_eq=A, b_eq=b, bounds=(0, None), method="highs")
    assert res.success, f"hull LP failed for {comp.reduced_formula}: {res.message}"
    return float(res.fun)

HULL_XCHECK = int(os.environ.get("HEO_HULL_XCHECK", 2))   # pymatgen cross-check on the first N compositions (0 = skip)
kT = W.K_B * W.T_SYNTH
rows = []
t0 = time.time()
for r in comps.itertuples():
    rec = R1.get(r.comp_id, {})
    if "O3_pris" not in rec or "P3_pris" not in rec: continue
    e = rec["O3_pris"]["energy"]; cnt = json.loads(r.site_counts)
    comp = Composition({**cnt, "Na": W.N_NA, "O": W.N_O})
    key = frozenset(str(el) for el in comp.elements)
    if MOCK_HULL:
        eh = W._mock_params(r.comp_id, cnt)["Ehull"]
    else:
        eh = max(0.0, e / comp.num_atoms - hull_energy_per_atom(comp, key))
        if len(rows) < HULL_XCHECK:
            _t = time.time()
            me = PDEntry(comp, e, name=r.comp_id)      # cell composition with the cell total energy;
            _sub = [entries[i] for i in _sub_cache[key][0]]     # pymatgen normalises per atom internally
            eh_pd = float(PhaseDiagram(_sub + [me]).get_e_above_hull(me))
            assert abs(eh_pd - eh) < 1e-6, f"LP hull {eh:.6f} != pymatgen {eh_pd:.6f} eV/atom for {r.comp_id}"
            print(f"  hull cross-check {r.comp_id}: LP {eh:.5f} == pymatgen {eh_pd:.5f} eV/atom "
                  f"({len(_sub)} competitors, PhaseDiagram took {time.time()-_t:.1f} s)", flush=True)
    s_atom = float(r.sconf_per_atom_kB)
    rows.append(dict(comp_id=r.comp_id, scheme=r.scheme, dopants=r.dopants, sconf_per_atom_kB=s_atom,
                     Ehull=eh, Ehull_eff=eh - kT*s_atom, T_star=eh/(W.K_B*s_atom),
                     dE_pris=(rec["P3_pris"]["energy"] - e)*1000/W.N_FU,
                     E_pris_O3=e, E_pris_P3=rec["P3_pris"]["energy"], Vol_pris_O3=rec["O3_pris"]["volume"]))
    if len(rows) % 500 == 0:
        HEARTBEAT.write_text(f"hull {len(rows)} {time.time():.0f}\n")     # keeps the idle watchdog off
        print(f"  Ehull {len(rows)} | {(time.time()-t0)/60:.1f} min", flush=True)
S = pd.DataFrame(rows)
tmp = f"{WORKDIR}/results/results_phase1.csv.tmp"
S.to_csv(tmp, index=False); os.replace(tmp, f"{WORKDIR}/results/results_phase1.csv")
print(f"Ehull for {len(S)} compositions in {(time.time()-t0)/60:.1f} min ({len(_sub_cache)} distinct element sets)")
print(S[["Ehull", "Ehull_eff", "T_star", "dE_pris"]].describe().round(3).to_string())
n_surv = int(((S.Ehull_eff < W.EHULL_EFF_CUT) & (S.dE_pris > W.DELTA_FLOOR_MEV_FU)).sum())
print(f"\nPASS 2 targets (Ehull_eff < {W.EHULL_EFF_CUT} AND pristine phase == O3): {n_surv} / {len(S)}")
for cut in (0.05, 0.10, 0.15):
    print(f"  Ehull_eff < {cut}: {int(((S.Ehull_eff < cut) & (S.dE_pris > W.DELTA_FLOOR_MEV_FU)).sum())}")


competing phases: 1119
HULL 200/1119 | 0.0 min
HULL 400/1119 | 0.0 min
HULL 600/1119 | 0.0 min
HULL 800/1119 | 0.0 min
HULL 1000/1119 | 0.0 min
HULL 1119/1119 | 0.0 min
hull entries: 1119 | skipped: 0
pass-1 coverage: 7073/7073
  hull cross-check A_Al-Cu-Co-Ca-V: LP 0.15124 == pymatgen 0.15124 eV/atom (179 competitors, PhaseDiagram took 0.7 s)
  hull cross-check A_Al-Cu-Ga-Ca-V: LP 0.14970 == pymatgen 0.14970 eV/atom (176 competitors, PhaseDiagram took 0.7 s)
  Ehull 500 | 0.0 min
  Ehull 1000 | 0.1 min
  Ehull 1500 | 0.1 min
  Ehull 2000 | 0.1 min
  Ehull 2500 | 0.1 min
  Ehull 3000 | 0.1 min
  Ehull 3500 | 0.2 min
  Ehull 4000 | 0.2 min
  Ehull 4500 | 0.2 min
  Ehull 5000 | 0.2 min
  Ehull 5500 | 0.2 min
  Ehull 6000 | 0.2 min
  Ehull 6500 | 0.3 min
  Ehull 7000 | 0.3 min
Ehull for 7073 compositions in 0.3 min (4055 distinct element sets)
          Ehull  Ehull_eff    T_star   dE_pris
count  7073.000   7073.000  7073.000  7073.000
mean      0.144      0.097  3613.426   132.641
std   

## 셀 9 — PASS 2 런처 (생존자 탈소듐, O3·P3에 같은 빈자리 패턴)

In [18]:
# ============ CELL 9: PASS 2 LAUNCHER ============
# Survivors (Ehull_eff < cut AND pristine phase == O3) are read from results_phase1.csv by the worker.
launch(2)

worker 0 -> GPU 0 (pid 878)

1 workers running (PASS 2). This cell returns now - watch with the next cell.


[<Popen: returncode: None args: ['/usr/local/bin/python', 'heo_worker.py', '-...>]

## 셀 10 — 모니터 (PASS 2)

In [19]:
# ============ CELL 10: MONITOR ============
monitor(2)

--- worker_0_p2.log ---
[w0] PASS-1 filter: 3854 / 7073 survive
[w0] PASS2 shard: 0 comps (model orb_v3_conservative_inf_mpa, engine torch-sim)
[w0] DONE in 0.00 h
--- worker_1_p2.log ---
Model Memory Estimation: Running forward pass on state with 8170 atoms and 86 systems.
[w1] 964/964 | 314 comps/h | ETA 0.0 h
[w1] DONE in 3.07 h
--- worker_2_p2.log ---
Model Memory Estimation: Running forward pass on state with 8170 atoms and 86 systems.
[w2] 963/963 | 305 comps/h | ETA 0.0 h
[w2] DONE in 3.16 h
--- worker_3_p2.log ---
Model Memory Estimation: Running forward pass on state with 8170 atoms and 86 systems.
[w3] 963/963 | 307 comps/h | ETA 0.0 h
[w3] DONE in 3.14 h

alive: 0/1 | exit codes: [0]
relaxed structures on disk: 53960
0, 0 %, 4 MiB

PASS 2 COMPLETE -> continue to the next cell


## 셀 10b — ★ 배리어: PASS 2 완료까지 대기 (Run All 안전장치)

In [ ]:
# ============ CELL 10b: BARRIER — block Run All until PASS 2 is complete ============
# Same as cell 7b: without this, "Run All" would let cell 11 judge phases from a partial PASS 2.
assert PROCS.get(2), "no PASS 2 workers tracked -> run the cell-9 launcher first"
monitor(2, wait=True)

## 셀 11 — Stage 3: 집계 · δ · 상 판정 · 채점 · 불변량 · audit

`dE_desod = min_k(E_P3,k − E_O3,k)` (짝지은 최소), `sigma_vac = std_k(E_P3,k − E_O3,k)`.
회색지대 `δ_i = max(σ_vac,i, |dE_A − dE_B|, 5)` — **대조 모델은 상 판정보다 반드시 앞에서** 읽습니다.
판정 뒤에 읽으면 δ를 갱신해도 `phase_desod`·`suppressed`·채점·저장 표·CIF가 전부 판정 이전 값으로 굳습니다.

불변량이 걸리면 실행은 멈추되 **진단용 표는 assert 앞에서 먼저 씁니다** — 계산은 못 믿어도 왜 걸렸는지는 봐야 합니다.
전부 통과하면 `_UNVALIDATED` 이름을 정식 이름으로 바꿉니다.

In [28]:
# ============ CELL 11: aggregate + judge + score + invariants + audit ============
import numpy as np, pandas as pd, json, hashlib, os, time
from pymatgen.core.periodic_table import Element

RULER = json.load(open(f"{WORKDIR}/results/ruler.json"))
assert RULER["model"] == W.MODEL_TAG, f"ruler built with {RULER['model']} != {W.MODEL_TAG}"
W.set_E_Na(RULER["E_Na"])

allE  = W.merged_energies()                                            # invariant 6 inside
comps = pd.read_csv(f"{WORKDIR}/results/compositions_master.csv").set_index("comp_id")
S     = pd.read_csv(f"{WORKDIR}/results/results_phase1.csv").set_index("comp_id")
COMP_HASH = hashlib.md5(open(f"{WORKDIR}/results/compositions_master.csv", "rb").read()).hexdigest()
survivors = S[(S.Ehull_eff < W.EHULL_EFF_CUT) & (S.dE_pris > W.DELTA_FLOOR_MEV_FU)].index

# ---- contrast model, loaded BEFORE the judgement ----
# Read after the judgement and delta would change nothing: phase_desod, suppressed, the score, the
# written tables and the exported CIFs would all stay frozen at their pre-contrast values.
CONTRAST = os.environ.get("HEO_CONTRAST_RESULTS", "")
DE_B = None
if CONTRAST and os.path.exists(CONTRAST):
    _B = pd.read_csv(CONTRAST, index_col="comp_id")
    assert "dE_desod" in _B.columns, f"{CONTRAST}: no dE_desod column -> not a results_v3.csv?"
    assert not _B.index.duplicated().any(), f"{CONTRAST}: duplicate comp_id"
    DE_B = _B["dE_desod"]
    print(f"contrast model: {int(DE_B.notna().sum())} compositions  <- {CONTRAST}")
else:
    print("no contrast results (set HEO_CONTRAST_RESULTS) - delta uses sigma_vac and the floor only")

need_tags = W.tags_for("all")
R2 = W.recs_by_comp(allE, survivors)
rows = [dict(W.metrics_from(rec), comp_id=cid)
        for cid, rec in ((c, R2.get(c, {})) for c in survivors)
        if all(t in rec for t in need_tags)]
M = pd.DataFrame(rows).set_index("comp_id")
print(f"complete compositions: {len(M)} / {len(survivors)} survivors")
assert len(M) >= 0.98 * max(len(survivors), 1), \
    f"PASS 2 incomplete: {len(M)}/{len(survivors)} survivors have a full tag set -> wait (cell 10b)"

floor = W.DELTA_FLOOR_MEV_FU
M["dE_desod_B"]     = DE_B.reindex(M.index) if DE_B is not None else np.nan
M["model_disagree"] = (M.dE_desod - M.dE_desod_B).abs()
M["delta"] = np.maximum.reduce([M.sigma_vac.values, M.model_disagree.fillna(0).values,
                                np.full(len(M), floor)])
M["phase_desod"] = [W.judge_phase(d, dl) for d, dl in zip(M.dE_desod, M.delta)]
M["dV_nat"]      = np.where(M.phase_desod == "O3", M.dV_O3, M.dV_trans)   # the volume of the phase that wins
M["suppressed"]  = (M.phase_desod == "O3")
print(M.phase_desod.value_counts().to_dict())
nf = int((M.n_phase_flip > 0).sum())
print(f"post-relax stacking: {len(M)-nf}/{len(M)} kept every label, {nf} have >=1 flipped structure")
if nf: print("  -> their dE_desod compares cells that are no longer O3 vs P3; treat with care.")
if DE_B is not None:
    print("model disagreement |dE_A - dE_B| (meV/f.u.):", M.model_disagree.describe().round(2).to_dict())

# 2026-09-04 (ranking): transition point by linear interpolation of dE(x) between x=1 and x=0.52.
# In the 0 K PBE currency essentially every composition is P3 at x=0.52 (first ORB run: 3,783 P3 /
# 70 AMB / 1 O3 of 3,854), exactly as both anchors -- including the experimentally delayed Zhao HEO --
# are.  "Retains O3 at 0.52" is therefore not a question this currency can answer; "who glides LAST"
# is.  x_cross is where dE crosses zero (smaller = more Na removed before gliding = better); it is the
# same quantity cell 5c reports for the calibration compounds, and it is what Zhao's "60 % of the
# capacity in O3" (x* ~ 0.71) measures.  ddE stays as a column but is no longer the score term:
# ddE = dE_desod - dE_pris punishes a large pristine O3 preference, which is the wrong direction.
X_DESOD = (W.N_NA - W.N_NA_REMOVE) / W.N_NA
def x_cross_of(dE_pris, dE_desod):
    drop = dE_pris - dE_desod
    return np.where(drop > 0, 1.0 - (1.0 - X_DESOD) * dE_pris / np.where(drop > 0, drop, 1.0), 0.0)
M["x_cross"] = x_cross_of(M.dE_pris.values, M.dE_desod.values)

# ---- capacity, ruler position, rank-sum score ----
def capacity_mAh_g(counts, n_Ni3, n_Mn3, n_Co2):
    """Redox electrons LEFT after the dopant accounting: Ni2+ -> Ni4+ is 2 e-, minus the sites already
    pushed to Ni3+ by low-valent dopants, plus Mn3+ -> Mn4+ and Co2+ -> Co3+.  V and Cu are not counted
    (undecided).  Capped at 1 e- per f.u. = one Na."""
    mass_fu = (Element("Na").atomic_mass + 2*Element("O").atomic_mass
               + sum(n*Element(e).atomic_mass for e, n in counts.items())/W.N_FU)
    e_avail = min(1.0, (2*counts.get("Ni", 0) - n_Ni3 + n_Mn3 + n_Co2)/W.N_FU)
    x_window = 1 - (W.N_NA - W.N_NA_REMOVE)/W.N_NA
    return 26801.0*e_avail/mass_fu, 26801.0*x_window/mass_fu

Rt = comps.join(S.drop(columns=[c for c in S.columns if c in comps.columns]), how="inner")
Rt = Rt.join(M.drop(columns=[c for c in M.columns if c in Rt.columns]), how="left")
Rt["cap_theor"], Rt["cap_window"] = zip(*[capacity_mAh_g(json.loads(r.site_counts), r.n_Ni3, r.n_Mn3, r.n_Co2)
                                          for r in Rt.itertuples()])
Rt["ruler_pos"] = ((Rt.dE_desod - RULER["dE_desod_HOST"])
                   / (RULER["dE_desod_HEO"] - RULER["dE_desod_HOST"]))      # 0 = HOST, 1 = HEO (Zhao 2020)
X_CROSS_HOST = float(x_cross_of(np.array([RULER["dE_pris_HOST"]]), np.array([RULER["dE_desod_HOST"]]))[0])
X_CROSS_HEO  = float(x_cross_of(np.array([RULER["dE_pris_HEO"]]),  np.array([RULER["dE_desod_HEO"]]))[0])
print(f"x_cross ruler: HOST {X_CROSS_HOST:.3f} | HEO {X_CROSS_HEO:.3f} | candidates median {Rt.x_cross.median():.3f}")

# 2026-09-04: signed volume changes of the two desodiated cells (diagnostic columns; the section-3
# quantities dV_O3 / dV_trans / dV_nat are unchanged).  Both cells shrink in-plane on TM oxidation; the
# prismatic gap of P3 (oxygen eclipsed across the gap) is wider than the octahedral gap of O3, so at the
# same x the P3 cell is the larger one and lands closer to the pristine volume.
_vol = allE.set_index(["comp_id", "tag"]).volume
Rt["Vol_desod_O3"] = [_vol.get((c, f"O3_desod_{int(k)}"), np.nan) if pd.notna(k) else np.nan
                      for c, k in zip(Rt.index, Rt.k_best)]
Rt["Vol_desod_P3"] = [_vol.get((c, f"P3_desod_{int(k)}"), np.nan) if pd.notna(k) else np.nan
                      for c, k in zip(Rt.index, Rt.k_best)]
Rt["sdV_O3"] = 100*(Rt.Vol_desod_O3 - Rt.Vol_pris_O3)/Rt.Vol_pris_O3
Rt["sdV_P3"] = 100*(Rt.Vol_desod_P3 - Rt.Vol_pris_O3)/Rt.Vol_pris_O3
_pair = Rt.dropna(subset=["Vol_desod_O3", "Vol_desod_P3"])
frac_P3_larger = float((_pair.Vol_desod_P3 > _pair.Vol_desod_O3).mean())
print(f"signed dV medians: O3 route {Rt.sdV_O3.median():+.2f}% | P3 route {Rt.sdV_P3.median():+.2f}% "
      f"| P3 cell larger than O3 cell at the same x: {frac_P3_larger:.2f} of {len(_pair)}")

# 2026-09-04 (ranking v2, user-approved): two tiers, one ordering function.
#   tier 0 = O3 retained at x=0.52 (suppressed) -- the screening's original question; always listed first.
#   tier 1 = everything else, ordered by rank-sum(x_cross asc, Ehull_eff asc).
# dV_nat is NO LONGER a score term: with every P3 composition it measures the accidental cancellation of
# c-expansion against a-contraction, and it penalised the one composition that actually kept O3 (real
# in-plane shrinkage 5.3 % vs ~1 % "cancelled" values).  It stays as a reported column.
SCORE_TERMS = {"x_cross": True, "Ehull_eff": True}   # True = ascending (lower is better)
def rank_sum(df):                        # ONE definition, used by the score and by the robustness check
    return sum(df[t].rank(ascending=asc) for t, asc in SCORE_TERMS.items())
def order(df):
    d = df.copy()
    d["score"] = rank_sum(d)
    d["tier"]  = np.where(d.suppressed == True, 0, 1)
    return d.sort_values(["tier", "score"])
cand = order(Rt.dropna(subset=["dE_desod"]))
top = cand.head(30)
n_tier0 = int((cand.tier == 0).sum())
print(f"candidates (judged): {len(cand)} | tier 0 (O3 retained at x=0.52): {n_tier0} -> always in top-30 "
      f"| tier-1 score range in top-30 {float(top[top.tier == 1].score.min()):.1f} - {float(top[top.tier == 1].score.max()):.1f}")

# ---- robustness ----
# Pessimistic shift: push dE_desod down by delta_mult*delta (the composition-specific noise), re-derive
# the phase, tier and x_cross, re-order.  Overlap with the base top-30 = how much of the list survives
# its own error bar.
def top_ids(df, delta_mult=1.0):
    d = df.dropna(subset=["dE_desod"]).copy()
    d["dE_desod"] = d.dE_desod - delta_mult*d.delta
    d["phase_desod"] = [W.judge_phase(x, dl) for x, dl in zip(d.dE_desod, d.delta)]
    d["suppressed"] = (d.phase_desod == "O3")
    d["x_cross"] = x_cross_of(d.dE_pris.values, d.dE_desod.values)
    d["dV_nat"] = np.where(d.phase_desod == "O3", d.dV_O3, d.dV_trans)
    return set(order(d).head(30).index)
base_top = set(top.index)
rob = {"n_candidates": len(cand), "n_tier0_suppressed_052": n_tier0, "n_AMB": int((Rt.phase_desod == "AMB").sum()),
       "delta_x1_overlap": len(base_top & top_ids(Rt, 1.0)) / max(1, len(base_top)),
       "delta_x2_overlap": len(base_top & top_ids(Rt, 2.0)) / max(1, len(base_top)),
       "inv4_frac_P3_cell_larger": frac_P3_larger,
       "x_cross_HOST": X_CROSS_HOST, "x_cross_HEO": X_CROSS_HEO,
       "top30_x_cross_range": [float(top.x_cross.min()), float(top.x_cross.max())]}
if DE_B is not None:
    rob["n_contrast"] = int(Rt.dE_desod_B.notna().sum())
    rob["model_disagree_median"] = float(Rt.model_disagree.median(skipna=True))
    _b = pd.read_csv(CONTRAST, index_col="comp_id")
    if "score" in _b.columns:
        rob["contrast_top30_overlap"] = len(base_top & set(_b.sort_values("score").head(30).index)) / max(1, len(base_top))
LEGACY = os.environ.get("HEO_LEGACY_TOP30", "")
if LEGACY and os.path.exists(LEGACY):
    leg = set(pd.read_csv(LEGACY).comp_id)
    rob["legacy_top30_overlap"] = len(base_top & leg) / max(1, len(leg))
print("robustness:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in rob.items()})

# ---- write the diagnostics FIRST, then assert ----
# A tripped invariant must stop the run, but it must not also destroy the evidence needed to fix it.
Rt = Rt.join(cand[["score", "tier"]], how="left")
Rt.to_csv(f"{WORKDIR}/results/results_v3_UNVALIDATED.csv")
top.to_csv(f"{WORKDIR}/results/top30_v3_UNVALIDATED.csv")
audit = dict(model=W.MODEL_TAG, compositions_md5=COMP_HASH, n_compositions=len(comps), n_pass1=len(S),
             n_survivors=len(survivors), n_complete=len(M), n_candidates=len(cand), n_tier0_suppressed_052=n_tier0,
             phase_desod_counts=Rt.phase_desod.value_counts(dropna=True).to_dict(),
             ruler=RULER, robustness=rob, ehull_eff_cut=W.EHULL_EFF_CUT, delta_floor=floor,
             sqs_cutoffs=W.SQS_CUTOFFS, sqs_steps=W.SQS_MC_STEPS, n_vac_samples=W.N_VAC_SAMPLES,
             dE_desod_definition="paired minimum  min_k(E_P3,k - E_O3,k)",
             sigma_vac_definition="std_k(E_P3,k - E_O3,k)",
             x_cross_definition="1 - (1 - 14/27) * dE_pris / (dE_pris - dE_desod)  (linear, 2-point)",
             score_definition="tier 0 = O3 retained at x=0.52 (always first); tier 1 ordered by rank-sum of "
                              "x_cross asc, Ehull_eff asc; dV_nat reported only (revised 2026-09-04 v2)",
             invariant4_definition="Vol(P3_desod,k_best) > Vol(O3_desod,k_best) for the majority (revised 2026-09-04)",
             lattice=dict(a=W.A_HEX, c=W.C_HEX, z_O=W.Z_O), E_Na=RULER["E_Na"],
             run_label=os.environ.get("HEO_RUN_LABEL", W.MODEL_TAG),
             orb_ckpt=os.environ.get("HEO_ORB_CKPT", ""),
             worker_md5=hashlib.md5(open(W.__file__, "rb").read()).hexdigest(),
             timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
json.dump(audit, open(f"{WORKDIR}/results/audit.json", "w"), indent=1, default=str)

V = Rt.dropna(subset=["V_avg"])
assert (V.V_avg > 0).all(), "invariant 1: negative voltage -> sign bug"
assert V.dE_desod.abs().max() < 500, "invariant 2: |dE| > 500 meV/f.u. -> per-cell instead of per-f.u.?"
# invariant 4 (revised 2026-09-04, user-approved).  The original "dV_trans >= dV_O3 for most P3
# compositions" assumed the transition adds net volume strain; the anchors (HOST 4.42 vs 1.84 %, HEO
# 2.61 vs 1.24 %) and the full ORB run show the opposite, for the reason given above.  What the
# invariant guards -- swapped O3/P3 labels or a volume-bookkeeping error -- is tested directly:
# with the same vacancy pattern the prismatic cell must be the larger one.
assert frac_P3_larger > 0.5, \
    f"invariant 4: the P3 cell should be larger than the O3 cell at the same x for most compositions (got {frac_P3_larger:.2f})"
assert COMP_HASH == hashlib.md5(open(f"{WORKDIR}/results/compositions_master.csv", "rb").read()).hexdigest(), \
    "invariant 7: compositions_master.csv changed during the run"
print("invariants 1,2,4,7 OK   (3 in cell 4, 5 in cell 5, 6 in every merged_energies call)")

for a, b in (("results_v3_UNVALIDATED.csv", "results_v3.csv"), ("top30_v3_UNVALIDATED.csv", "top30_v3.csv")):
    os.replace(f"{WORKDIR}/results/{a}", f"{WORKDIR}/results/{b}")
print("written:", sorted(os.listdir(f"{WORKDIR}/results")))
print(top[["tier", "scheme", "V_avg", "dE_pris", "dE_desod", "x_cross", "phase_desod", "Ehull_eff", "T_star",
           "dV_nat", "cap_theor"]].round(3).head(10).to_string())


no contrast results (set HEO_CONTRAST_RESULTS) - delta uses sigma_vac and the floor only
complete compositions: 3854 / 3854 survivors
{'P3': 3783, 'AMB': 70, 'O3': 1}
post-relax stacking: 3843/3854 kept every label, 11 have >=1 flipped structure
  -> their dE_desod compares cells that are no longer O3 vs P3; treat with care.
x_cross ruler: HOST 0.570 | HEO 0.607 | candidates median 0.632
signed dV medians: O3 route -3.61% | P3 route -1.16% | P3 cell larger than O3 cell at the same x: 0.94 of 3854
candidates (judged): 3854 | tier 0 (O3 retained at x=0.52): 1 -> always in top-30 | tier-1 score range in top-30 112.0 - 454.0
robustness: {'n_candidates': 3854, 'n_tier0_suppressed_052': 1, 'n_AMB': 70, 'delta_x1_overlap': 0.833, 'delta_x2_overlap': 0.733, 'inv4_frac_P3_cell_larger': 0.941, 'x_cross_HOST': 0.57, 'x_cross_HEO': 0.607, 'top30_x_cross_range': [0.49435313845062645, 0.5968646741695858]}
invariants 1,2,4,7 OK   (3 in cell 4, 5 in cell 5, 6 in every merged_energies call)
written: ['

## 셀 12 — top-N 내보내기 (DFT Tier-1 입력)

`HEO_SAVE_RELAXED=1`이라 이완 구조 CIF는 계산할 때 **이미 저장돼 있습니다.** 여기서는 다시 이완하지 않고 색인만 만듭니다 —
재이완하면 캐시 에너지와 미세하게 달라져 저장된 표와 내보낸 구조가 어긋납니다.
`k_best`는 `dE_desod`를 결정한 바로 그 빈자리 샘플이므로, O3와 P3가 **같은 Na 배열**을 쓴 짝을 내보냅니다.

In [29]:
# ============ CELL 12: export the top-N structures (DFT Tier-1 input) ============
# HEO_SAVE_RELAXED=1 means every relaxed cell was already written to CIF while it was computed, so
# nothing is re-relaxed here.  Re-relaxing would produce slightly different energies than the ones
# already in results_v3.csv, and the exported structures would no longer match the reported numbers.
import shutil, json, os
import numpy as np, pandas as pd
top = pd.read_csv(f"{WORKDIR}/results/top30_v3.csv", index_col=0)
OUT = f"{WORKDIR}/results/top30_cifs"; os.makedirs(OUT, exist_ok=True)
index, missing = {}, []
for cid in top.index:
    kb = int(top.loc[cid, "k_best"])          # the vacancy sample that set dE_desod: O3 and P3 share it
    files = {"O3_pris": f"{cid}__O3_pris.cif", "P3_pris": f"{cid}__P3_pris.cif",
             "O3_desod": f"{cid}__O3_desod_{kb}.cif", "P3_desod": f"{cid}__P3_desod_{kb}.cif"}
    got = {}
    for role, fn in files.items():
        src = f"{W.RELAXED_DIR}/{fn}"
        if os.path.exists(src): shutil.copy(src, f"{OUT}/{fn}"); got[role] = fn
        else: missing.append(f"{cid}/{role}")
    index[cid] = dict(k_best=kb, files=got, **{c: (float(top.loc[cid, c]) if c in top.columns else None)
                      for c in ("dE_desod", "ddE", "Ehull_eff", "T_star", "V_avg", "dV_nat",
                                "ruler_pos", "cap_theor", "delta", "sigma_vac")})
json.dump(index, open(f"{WORKDIR}/results/top30_cif_index.json", "w"), indent=1)
print(f"exported {len(index)} compositions x 4 cells -> {OUT}")
if missing:
    print("MISSING:", missing[:10], "..." if len(missing) > 10 else "")
    print("  (HEO_SAVE_RELAXED was off for part of the run; re-run the launcher for those compositions)")

# ORB confidence head: the batch engine cannot read it, so take a single point for the top-N only.
if MODE == "orb":
    from pymatgen.core import Structure as _St
    from pymatgen.io.ase import AseAtomsAdaptor
    from orb_models.forcefield import pretrained
    try:
        from orb_models.forcefield.inference.calculator import ORBCalculator   # orb-models >= 0.6
    except ImportError:
        from orb_models.forcefield.calculator import ORBCalculator             # <= 0.5 layout
    _l = getattr(pretrained, W.ORB_MODEL)(device=W.DEVICE)
    _raw, _ad = _l if isinstance(_l, tuple) else (_l, None)
    try:
        _calc = ORBCalculator(_raw, device=W.DEVICE) if _ad is None else \
                ORBCalculator(_raw, atoms_adapter=_ad, device=W.DEVICE)
    except TypeError:
        _calc = ORBCalculator(_raw, device=W.DEVICE)
    conf = {}
    for cid, meta in index.items():
        fn = meta["files"].get("O3_desod")
        if not fn: continue
        at = AseAtomsAdaptor.get_atoms(_St.from_file(f"{OUT}/{fn}")); at.calc = _calc
        at.get_potential_energy(); c = at.calc.results.get("confidence")
        conf[cid] = (float(np.mean(c)), float(np.min(c))) if c is not None else (float("nan"),)*2
    if conf:
        top["conf_mean"] = [conf.get(c, (np.nan, np.nan))[0] for c in top.index]
        top["conf_min"]  = [conf.get(c, (np.nan, np.nan))[1] for c in top.index]
        top.to_csv(f"{WORKDIR}/results/top30_v3.csv")
        print("confidence head read for", len(conf), "compositions |",
              "min conf_min =", round(float(np.nanmin(top.conf_min)), 3))

exported 30 compositions x 4 cells -> /workspace/heo_v2/results/top30_cifs


/usr/local/lib/python3.12/dist-packages/orb_models/common/training/util.py:20: UserWarning: Setting global torch default dtype to torch.float32.
  warnings.warn(f"Setting global torch default dtype to {torch_dtype}.")
/usr/local/lib/python3.12/dist-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/usr/local/lib/python3.12/dist-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]


confidence head read for 30 compositions | min conf_min = 0.0


## 셀 12b — (선택) base ↔ 현재 실행 대조 (파인튜닝 A/B)

파인튜닝한 ORB로 재실행할 때 **별도 `HEO_WORKDIR`** 을 쓰고, `HEO_ORB_CKPT=<가중치 파일>`과
`HEO_BASE_RESULTS=<base 실행의 results_v3.csv>` 를 export 하면 이 셀이 두 실행을 조성 단위로 짝지어 비교합니다.
구조 생성이 내용 주소(comp_rng) 결정론이라 **두 실행은 완전히 같은 SQS 셀·같은 빈자리 패턴**을 이완합니다 —
아래 차이는 전부 모델 차이입니다.

찍는 것: 부호 일치율(§8 "부호 일치 측정"), base 대비 기울기(§8 "연화 계수" — 1보다 크면 파인튜닝이 에너지 차이를 키움),
Spearman 순위 상관(랭킹 보존), phase 판정 혼동 행렬, suppressed·top-30(ddE 기준) 겹침, 앵커 gap 변화.
결과는 `results/compare_base.{csv,json}`에 저장되어 셀 13 아카이브에 포함됩니다. base 실행에서는 그냥 스킵됩니다.

추가 기록(2026-09-03): 부호 있는 평균 이동(`mean_shift`), phase 뒤집힘 원인 분해(부호 vs δ), scheme별 일치율,
실제 `top30_v3.csv`(rank-sum 점수) 겹침, 앵커 HOST/HEO `dE_desod`·`E_Na` 상세.


In [ ]:
# ============ CELL 12b: base-vs-current run comparison (fine-tuning A/B) ============
# Optional.  HEO_BASE_RESULTS points at a previous run's results_v3.csv (e.g. stock ORB); the
# current run must live in its OWN HEO_WORKDIR with a distinct MODEL_TAG (HEO_ORB_CKPT sets one).
# comp_rng() content-addresses SQS and vacancy patterns, so both runs relaxed identical inputs:
# every difference measured here is the model, not the sampling.
import os, json
import numpy as np, pandas as pd

BASE = os.environ.get("HEO_BASE_RESULTS", "")
if not (BASE and os.path.exists(BASE)):
    print("no base run to compare (export HEO_BASE_RESULTS=<base results_v3.csv>) - skipping")
else:
    cur  = pd.read_csv(f"{WORKDIR}/results/results_v3.csv", index_col="comp_id")
    base = pd.read_csv(BASE, index_col="comp_id")
    both = cur.join(base, how="inner", rsuffix="_base")
    print(f"current {len(cur)}  x  base {len(base)}  ->  common: {len(both)} compositions")
    assert len(both), "no common compositions -> wrong HEO_BASE_RESULTS?"

    stats = {"n_current": len(cur), "n_base": len(base), "n_common": len(both),
             "model_current": W.MODEL_TAG, "base_results": BASE}
    for q in ("dE_pris", "dE_desod", "ddE", "V_avg", "Ehull_eff"):
        if q not in both or f"{q}_base" not in both: continue
        a, b = both[q], both[f"{q}_base"]
        ok = a.notna() & b.notna()
        if not ok.any(): continue
        stats[q] = dict(
            slope_vs_base=round(float((a[ok]*b[ok]).sum() / max((b[ok]**2).sum(), 1e-12)), 3),
            pearson=round(float(a[ok].corr(b[ok])), 3),
            spearman=round(float(a[ok].corr(b[ok], method="spearman")), 3),
            sign_agree=round(float((np.sign(a[ok]) == np.sign(b[ok])).mean()), 3),
            mean_shift=round(float((a[ok] - b[ok]).mean()), 2),      # signed: which WAY the model moved
            mean_abs_shift=round(float((a[ok] - b[ok]).abs().mean()), 2))
    # slope > 1: the current model AMPLIFIES energy differences vs base (de-softening);
    # spearman ~ 1 with slope != 1: ranking preserved, only the scale moved (Deng 2025 picture).

    # per-scheme sign agreement: uneven fine-tuning coverage shows up here before it shows up
    # in the global mean (scheme C is small and chemically distinct - watch it).
    if "scheme" in both and "dE_desod_base" in both:
        per = {}
        for s, g in both.groupby("scheme"):
            a, b = g.dE_desod, g.dE_desod_base
            ok = a.notna() & b.notna()
            if ok.sum() < 5: continue
            per[str(s)] = dict(n=int(ok.sum()),
                               sign_agree=round(float((np.sign(a[ok]) == np.sign(b[ok])).mean()), 3),
                               slope_vs_base=round(float((a[ok]*b[ok]).sum() / max((b[ok]**2).sum(), 1e-12)), 3))
        stats["dE_desod_by_scheme"] = per

    conf = pd.crosstab(both.phase_desod, both.phase_desod_base, dropna=False)
    print("\nphase_desod confusion (rows=current, cols=base):\n", conf.to_string())
    stats["phase_confusion"] = {f"{r}|{c}": int(conf.loc[r, c]) for r in conf.index for c in conf.columns}

    # why did a judgement flip: the dE SIGN changed (the models disagree on the physics), or the
    # sign held and only delta moved the AMB boundary (uncertainty bookkeeping, not physics)?
    okp  = both.phase_desod.notna() & both.phase_desod_base.notna()
    flip = okp & (both.phase_desod != both.phase_desod_base)
    sgn  = np.sign(both.dE_desod) != np.sign(both.dE_desod_base)
    stats["phase_flips"] = dict(total=int(flip.sum()),
                                sign_driven=int((flip & sgn).sum()),
                                delta_driven=int((flip & ~sgn).sum()))

    sup_c = set(both.index[both.suppressed == True]); sup_b = set(both.index[both.suppressed_base == True])
    stats["suppressed"] = dict(current=len(sup_c), base=len(sup_b),
                               gained=len(sup_c - sup_b), lost=len(sup_b - sup_c),
                               jaccard=round(len(sup_c & sup_b) / max(len(sup_c | sup_b), 1), 3))
    topN = 30
    t_c = set(both.ddE.nlargest(topN).index); t_b = set(both.ddE_base.nlargest(topN).index)
    stats[f"top{topN}_ddE_overlap"] = len(t_c & t_b)

    # the DFT-bound list is the rank-sum top-30 (cell 11), not the ddE top-30: compare the
    # actually saved files so this number answers "how many Tier-1 candidates change?"
    bt = os.path.join(os.path.dirname(BASE), "top30_v3.csv")
    ct = f"{WORKDIR}/results/top30_v3.csv"
    if os.path.exists(bt) and os.path.exists(ct):
        s_b = set(pd.read_csv(bt).comp_id); s_c = set(pd.read_csv(ct).comp_id)
        stats["top30_score_overlap"] = len(s_b & s_c)

    br = os.path.join(os.path.dirname(BASE), "ruler.json")
    if os.path.exists(br):
        rb = json.load(open(br)); rc = json.load(open(f"{WORKDIR}/results/ruler.json"))
        g = lambda d, k: round(float(d.get(k, float("nan"))), 2)
        stats["anchor_gap"] = dict(base=g(rb, "gap"), current=g(rc, "gap"),
                                   dE_desod_HOST=[g(rb, "dE_desod_HOST"), g(rc, "dE_desod_HOST")],
                                   dE_desod_HEO=[g(rb, "dE_desod_HEO"), g(rc, "dE_desod_HEO")],
                                   E_Na=[round(float(rb.get("E_Na", float("nan"))), 4),
                                         round(float(rc.get("E_Na", float("nan"))), 4)],
                                   base_model=rb.get("model"), current_model=rc.get("model"))

    keep = [c for c in ("scheme", "dE_pris", "dE_pris_base", "dE_desod", "dE_desod_base",
                        "ddE", "ddE_base", "delta", "delta_base", "phase_desod", "phase_desod_base",
                        "suppressed", "suppressed_base", "V_avg", "V_avg_base",
                        "Ehull_eff", "Ehull_eff_base", "ruler_pos", "ruler_pos_base") if c in both]
    both[keep].to_csv(f"{WORKDIR}/results/compare_base.csv")
    json.dump(stats, open(f"{WORKDIR}/results/compare_base.json", "w"), indent=1, default=str)
    print("\ncomparison:", json.dumps({k: v for k, v in stats.items()
                                        if k not in ("phase_confusion", "base_results")}, indent=1, default=str))
    print("written: results/compare_base.csv, results/compare_base.json")


## 셀 13 — 아카이브하고 팟 종료 (마지막)

순서가 중요합니다: 결과 → 아카이브 → (업로드) → `DONE` → 종료.
`/workspace`가 Network Volume이므로 `delete`도 안전합니다(데이터는 볼륨에 남습니다). 그래도 기본은 `stop`으로 두었으니 삭제까지 원하시면 `HEO_TERMINATE_MODE=delete`를 export하세요.

In [24]:
# ============ CELL 13: archive + terminate ============
# Order matters: results -> archive -> (upload) -> DONE sentinel -> terminate.
# Every external binary is optional; a missing one must never skip the REST fallback, or the pod bills on.
import tarfile, subprocess, shutil, urllib.request, json, time, os

def _run(cmd, **kw):
    try: return subprocess.run(cmd, **kw)
    except (FileNotFoundError, PermissionError, OSError) as e:
        print(f"  {cmd[0]} unavailable: {e!r}"); return None

def pod_terminate(mode):
    pod = os.environ.get("RUNPOD_POD_ID")
    if not pod: return False, "RUNPOD_POD_ID not set (nothing to terminate)"
    cmds = ([["runpodctl", "pod", "delete", pod], ["runpodctl", "remove", "pod", pod]] if mode == "delete"
            else [["runpodctl", "pod", "stop", pod], ["runpodctl", "stop", "pod", pod]])
    for c in cmds:
        r = _run(c, capture_output=True)
        if r is not None and r.returncode == 0: return True, " ".join(c)
    key = os.environ.get("RUNPOD_API_KEY")
    if not key: return False, "runpodctl failed and RUNPOD_API_KEY not set"
    url = f"https://rest.runpod.io/v1/pods/{pod}" + ("" if mode == "delete" else "/stop")
    req = urllib.request.Request(url, method="DELETE" if mode == "delete" else "POST",
                                 headers={"Authorization": f"Bearer {key}"})
    try:
        urllib.request.urlopen(req, timeout=20); return True, "REST " + url
    except Exception as e:
        return False, f"REST failed: {e!r}"

mode = os.environ.get("HEO_TERMINATE_MODE", "stop")
tgz = f"{WORKDIR}/heo_v3_results_{time.strftime('%Y%m%d_%H%M')}.tgz"
with tarfile.open(tgz, "w:gz") as tf:
    for sub in ("results", "checkpoints", "hull_cache"):
        if os.path.exists(f"{WORKDIR}/{sub}"): tf.add(f"{WORKDIR}/{sub}", arcname=sub)
print("archive:", tgz, f"{os.path.getsize(tgz)/1e6:.1f} MB")
print("  (structures/ is left out: it is large and stays on the Network Volume)")

uploaded = False
remote = os.environ.get("HEO_RCLONE_REMOTE")
if remote and shutil.which("rclone"):
    r = _run(["rclone", "copy", tgz, remote]); uploaded = (r is not None and r.returncode == 0)
    print("uploaded:", uploaded)

hook = os.environ.get("HEO_WEBHOOK")
if hook:
    try:
        urllib.request.urlopen(urllib.request.Request(
            hook, data=json.dumps({"text": f"[HEO v3] run finished, terminate={mode}, uploaded={uploaded}"}).encode(),
            headers={"Content-Type": "application/json"}), timeout=10)
    except Exception as e:
        print("webhook failed:", e)

open(f"{WORKDIR}/results/DONE", "w").write(json.dumps({"mode": mode, "uploaded": uploaded, "t": time.time()}))
print("terminate mode:", mode, "| pod:", os.environ.get("RUNPOD_POD_ID"))
if mode in ("delete", "stop"):
    time.sleep(5)                                   # let file writes flush
    ok, how = pod_terminate(mode)
    print(f"pod {mode}: {'issued via ' + how if ok else 'FAILED -> ' + how}")
    if not ok:
        print("  !! the cell-1b watchdog retries in ~5 min; if that also fails, stop the pod in the console")
else:
    print("HEO_TERMINATE_MODE=none -> pod left RUNNING and BILLING. Stop it yourself when done.")

archive: /workspace/heo_v2/heo_v3_results_20260904_0205.tgz 4.4 MB
  (structures/ is left out: it is large and stays on the Network Volume)
terminate mode: none | pod: ymyn7pgvmswibk
HEO_TERMINATE_MODE=none -> pod left RUNNING and BILLING. Stop it yourself when done.


## RunPod에서 실행

```bash
pip install pymatgen mp-api icet ase torch-sim-atomistic orb-models   # 대조 실행은 mace-torch 추가
export HEO_WORKDIR=/workspace/heo_v2      # Network Volume
export HEO_MODE=orb
export MP_API_KEY=...                     # 셸에서만. 노트북에 적지 마세요
export HEO_TERMINATE_MODE=none            # 기본 none: 워치독은 로그만, 팟은 직접 stop\n# 파인튜닝 A/B: export HEO_ORB_CKPT=...   HEO_WORKDIR=/workspace/heo_v2_ft   HEO_BASE_RESULTS=/workspace/heo_v2/results/results_v3.csv
# 선택: export HEO_RCLONE_REMOTE=gdrive:heo_v2   HEO_WEBHOOK=...
```

`heo_worker.py`를 노트북과 같은 디렉터리에 올린 뒤 셀 1부터 순서대로.

**첫 실행은 스모크로**: `export HEO_LIMIT=30 HEO_TERMINATE_MODE=none` → 셀 1~11 완주 확인
(로그에 `[SQS-FALLBACK]` 없음 / hull 경쟁상 수 / 앵커 표 / `sigma_paired` vs `sigma_max`). 그다음 `unset HEO_LIMIT` 하고 전체.

**대조 실행**: `HEO_MODE=mace HEO_WORKDIR=/workspace/heo_v2_mace` (WORKDIR 분리 필수)로 한 번 더 돌린 뒤
`export HEO_CONTRAST_RESULTS=/workspace/heo_v2_mace/results/results_v3.csv` 하고 셀 11만 다시 실행하면 δ에 반영됩니다.

## 다음 (이 노트북 밖)
* `top30_v3.csv` + `top30_cifs/`에 대한 DFT 앵커 + Tier-1 single point → 연화 계수, 부호 일치율 → 파인튜닝 여부 결정
* top 조성: 4점 x 격자(1.0 / 0.85 / 0.67 / 0.52), Kaufman(PRM 2019) 질서 빈자리 패턴
* DFT 실행 전 수렴 테스트(GGA+U, k-point)

## RunPod에서 실행

```bash
pip install pymatgen mp-api icet ase torch-sim-atomistic orb-models   # 대조 실행은 mace-torch 추가
export HEO_WORKDIR=/workspace/heo_v2      # Network Volume
export HEO_MODE=orb
export MP_API_KEY="0md9oV9ieH21W4cmejXD6YQZHz6rRv7F"                     # 셸에서만. 노트북에 적지 마세요
export HEO_TERMINATE_MODE=none            # 기본 none: 워치독은 로그만, 팟은 직접 stop\n# 파인튜닝 A/B: export HEO_ORB_CKPT=...   HEO_WORKDIR=/workspace/heo_v2_ft   HEO_BASE_RESULTS=/workspace/heo_v2/results/results_v3.csv
# 선택: export HEO_RCLONE_REMOTE=gdrive:heo_v2   HEO_WEBHOOK=...
```

`heo_worker.py`를 노트북과 같은 디렉터리에 올린 뒤 셀 1부터 순서대로.

**첫 실행은 스모크로**: `export HEO_LIMIT=30 HEO_TERMINATE_MODE=none` → 셀 1~11 완주 확인
(로그에 `[SQS-FALLBACK]` 없음 / hull 경쟁상 수 / 앵커 표 / `sigma_paired` vs `sigma_max`). 그다음 `unset HEO_LIMIT` 하고 전체.

**대조 실행**: `HEO_MODE=mace HEO_WORKDIR=/workspace/heo_v2_mace` (WORKDIR 분리 필수)로 한 번 더 돌린 뒤
`export HEO_CONTRAST_RESULTS=/workspace/heo_v2_mace/results/results_v3.csv` 하고 셀 11만 다시 실행하면 δ에 반영됩니다.

## 다음 (이 노트북 밖)
* `top30_v3.csv` + `top30_cifs/`에 대한 DFT 앵커 + Tier-1 single point → 연화 계수, 부호 일치율 → 파인튜닝 여부 결정
* top 조성: 4점 x 격자(1.0 / 0.85 / 0.67 / 0.52), Kaufman(PRM 2019) 질서 빈자리 패턴
* DFT 실행 전 수렴 테스트(GGA+U, k-point)